In [ ]:
import json
import re
from pathlib import Path
from collections import Counter


# ============================================================
# 1. 단위 / 이름 정규화 설정
# ============================================================

NUM = r"(?:\d+\s+\d+/\d+|\d+/\d+|\d+(?:\.\d+)?)"
RANGE = rf"(?P<a>{NUM})(?:\s*[~\-–]\s*(?P<b>{NUM}))?"

UNIT_PATTERN = (
    r"큰술|작은술|티스푼|스푼|Ts|TS|ts|T|t|"
    r"kg|KG|g|G|ml|mL|ML|L|l|"
    r"종이컵|소주잔|컵|개|마리|대|쪽|줌|꼬집|팩|봉지|장|톨|"
    r"뿌리|통|인분|움큼|주먹|모|알|스틱|캔|병|포기"
)

MEASURE_RE = re.compile(
    rf"(?<!\w){RANGE}\s*(?P<unit>{UNIT_PATTERN})(?!\w)"
)

QUAL_RE = re.compile(
    r"\s*(약간|조금|적당량|듬뿍|톡톡|취향껏|적당히)\s*$"
)


UNIT_ALIASES = {
    "T": "큰술",
    "Ts": "큰술",
    "TS": "큰술",
    "스푼": "큰술",

    "t": "작은술",
    "ts": "작은술",
    "티스푼": "작은술",

    "mL": "ml",
    "ML": "ml",

    "G": "g",
    "KG": "kg",

    "l": "L",
}


# 같은 식재료인데 표현만 다른 것
ALIASES = {
    "달걀": "계란",

    "고추가루": "고춧가루",

    "후추가루": "후추",
    "후춧가루": "후추",

    "소세지": "소시지",

    "굴 소스": "굴소스",
    "토마토 소스": "토마토소스",

    "올리브 오일": "올리브유",

    "깨소금": "깨",
}


# 재료 배열에 들어가 있더라도
# 실제로는 양념으로 분류할 목록
SEASONING_NAMES = {
    "간장",
    "진간장",
    "국간장",
    "양조간장",
    "맛간장",

    "고추장",
    "된장",
    "쌈장",

    "굴소스",
    "돈까스소스",
    "토마토소스",
    "케첩",
    "마요네즈",

    "고춧가루",

    "설탕",
    "황설탕",
    "소금",
    "후추",
    "식초",

    "맛술",
    "미림",
    "청주",

    "요리당",
    "물엿",
    "올리고당",
    "매실액",
    "꿀",

    "참기름",
    "들기름",
    "식용유",
    "올리브유",
    "고추기름",

    "액젓",
    "멸치액젓",
    "까나리액젓",

    "참깨",
    "깨",
}


# 조리 단계에서 "명시적으로" 등장한 경우만 추출
TOOL_PATTERNS = [
    (r"에어프라이어|에어프라이러", "에어프라이어"),

    (r"전기\s*밥솥", "전기밥솥"),
    (r"압력\s*밥솥", "압력밥솥"),
    (r"압력\s*솥", "압력솥"),

    (r"전자\s*레인지|전자렌지", "전자레인지"),

    (r"궁중\s*팬", "궁중팬"),
    (r"후라이팬|프라이팬", "프라이팬"),

    (r"뚝배기", "뚝배기"),
    (r"냄비", "냄비"),
    (r"오븐", "오븐"),

    (r"믹서기", "믹서기"),
    (r"블렌더", "블렌더"),

    (r"찜기", "찜기"),
    (r"그릴", "그릴"),
    (r"토스터", "토스터"),

    # 그냥 "팬"이라고 적은 경우
    (r"\b팬\b|팬에|팬을|팬으로|팬에서", "프라이팬"),
]


# ============================================================
# 2. 숫자 처리
# ============================================================

def to_number(text):

    if text is None:
        return None

    text = text.strip()

    # 1 1/2 같은 값
    if " " in text and "/" in text:

        whole, fraction = text.split(None, 1)

        numerator, denominator = fraction.split("/")

        return (
            float(whole)
            + float(numerator) / float(denominator)
        )

    # 1/2
    if "/" in text:

        numerator, denominator = text.split("/")

        return (
            float(numerator)
            / float(denominator)
        )

    return float(text)


def measure_to_dict(match):

    amount_min = to_number(
        match.group("a")
    )

    if match.group("b"):

        amount_max = to_number(
            match.group("b")
        )

    else:

        amount_max = amount_min


    unit_raw = match.group("unit")

    unit_normalized = UNIT_ALIASES.get(
        unit_raw,
        unit_raw
    )


    return {
        "amount_text":
            match.group(0).strip(),

        "amount_min":
            amount_min,

        "amount_max":
            amount_max,

        "unit_raw":
            unit_raw,

        "unit_normalized":
            unit_normalized,
    }


# ============================================================
# 3. 재료 이름 정규화
# ============================================================

def normalize_name_and_detail(name):

    name = (
        name
        or ""
    ).strip()

    name = re.sub(
        r"\s+",
        " ",
        name
    )

    # 혹은 → 또는
    name = name.replace(
        "혹은",
        "또는"
    )


    preparation = []

    detail_parts = []


    # --------------------------------------------------------
    # 다진마늘 / 다진 마늘 → 마늘
    # --------------------------------------------------------

    preparation_rules = [

        (
            r"^다진\s*(마늘|양파|파|대파|생강)$",
            "다진"
        ),

        (
            r"^간\s*(마늘|양파|생강)$",
            "간"
        ),

        (
            r"^통\s*(마늘)$",
            "통"
        ),

        (
            r"^불린\s*(찹쌀|쌀|미역|당면)$",
            "불린"
        ),

        (
            r"^마른\s+(.+)$",
            "마른"
        ),

        (
            r"^고운\s*(고춧가루|고추가루)$",
            "고운"
        ),
    ]


    for pattern, prep in preparation_rules:

        match = re.match(
            pattern,
            name
        )

        if match:

            name = match.group(1)

            preparation.append(
                prep
            )

            break


    # 생닭 → 닭
    if name == "생닭":

        name = "닭"

        preparation.append(
            "생"
        )


    # --------------------------------------------------------
    # 돼지고기 찌개용 또는 목살
    #
    # Ingredient = 돼지고기
    # detail = 찌개용 또는 목살
    # --------------------------------------------------------

    descriptor_patterns = [

        r"^(돼지고기)\s+(.+)$",

        r"^(소고기)\s+(.+)$",

        r"^(오징어)\s+(큰\s*사이즈|대|중|소)$",

        r"^(모짜렐라치즈)\s+(듬뿍|적당량|약간)$",
    ]


    for pattern in descriptor_patterns:

        match = re.match(
            pattern,
            name
        )

        if match:

            name = match.group(1)

            detail_parts.append(
                match.group(2).strip()
            )

            break


    # --------------------------------------------------------
    # 일반적인 용도 표현
    # --------------------------------------------------------

    match = re.match(
        r"^(.+?)\s+"
        r"(국거리용|"
        r"찌개용(?:\s+또는\s+.+)?|"
        r"큰\s*사이즈|"
        r"듬뿍)$",
        name
    )


    if match:

        name = match.group(1).strip()

        detail_parts.append(
            match.group(2).strip()
        )


    # alias 정규화
    name = ALIASES.get(
        name,
        name
    )


    name = re.sub(
        r"\s+",
        " ",
        name
    ).strip()


    detail = "; ".join(
        detail_parts
    )


    return (
        name,
        detail,
        preparation
    )


# ============================================================
# 4. 재료 / 양념 분류
# ============================================================

def classify_item(
    name,
    group="",
    original_type="ingredient"
):

    group_text = (
        group
        or ""
    ).replace(
        " ",
        ""
    )


    # 원래 seasonings 배열에 있었으면 양념
    if original_type == "seasoning":

        return "seasoning"


    # 그룹명에 양념/소스가 있으면 양념
    if any(
        keyword in group_text

        for keyword in [
            "양념",
            "소스",
            "드레싱"
        ]
    ):

        return "seasoning"


    # 이름으로 판단
    if name in SEASONING_NAMES:

        return "seasoning"


    return "ingredient"


# ============================================================
# 5. raw 재료 한 개 다시 파싱
# ============================================================

def parse_food_item(
    raw,
    group="",
    original_type="ingredient"
):

    raw = (
        raw
        or ""
    ).replace(
        "구매",
        ""
    ).strip()


    raw = re.sub(
        r"\s+",
        " ",
        raw
    )


    # raw 안의 모든 숫자 + 단위 검색
    measurements = list(
        MEASURE_RE.finditer(
            raw
        )
    )


    qualitative_amount = None


    # --------------------------------------------------------
    # 숫자 단위가 존재
    # --------------------------------------------------------

    if measurements:

        # 마지막 값을 대표 양으로 사용
        #
        # 미역 20g 1줌
        #
        # 대표 = 1줌
        # secondary = 20g

        primary_match = (
            measurements[-1]
        )


        primary = measure_to_dict(
            primary_match
        )


        secondary_amounts = [

            measure_to_dict(
                match
            )

            for match
            in measurements[:-1]
        ]


        # 첫 숫자 앞까지를 재료명으로
        name_text = raw[
            :measurements[0].start()
        ].strip(
            " ,/"
        )


        # 중간 설명
        if len(measurements) > 1:

            middle_text = raw[
                measurements[0].end()
                :
                primary_match.start()
            ].strip(
                " ,/"
            )

        else:

            middle_text = ""


        trailing_text = raw[
            primary_match.end():
        ].strip(
            " ,/"
        )


        detail_extra = " ".join(
            x
            for x
            in [
                middle_text,
                trailing_text
            ]
            if x
        )


    # --------------------------------------------------------
    # 약간 / 조금 등
    # --------------------------------------------------------

    else:

        qual_match = QUAL_RE.search(
            raw
        )


        if qual_match:

            qualitative_amount = (
                qual_match.group(1)
            )


            name_text = raw[
                :qual_match.start()
            ].strip(
                " ,/"
            )

        else:

            name_text = raw


        primary = {

            "amount_text":
                qualitative_amount or "",

            "amount_min":
                None,

            "amount_max":
                None,

            "unit_raw":
                None,

            "unit_normalized":
                None,
        }


        secondary_amounts = []

        detail_extra = ""


    # --------------------------------------------------------
    # 이름 정규화
    # --------------------------------------------------------

    (
        name_normalized,
        detail,
        preparation

    ) = normalize_name_and_detail(
        name_text
    )


    if detail_extra:

        detail = "; ".join(
            x
            for x
            in [
                detail,
                detail_extra
            ]
            if x
        )


    item_type = classify_item(
        name_normalized,
        group,
        original_type
    )


    return {

        # 원래 이름
        "name_raw":
            name_text,

        # Neo4j Ingredient 노드 이름으로 사용
        "name_normalized":
            name_normalized,

        # 국거리용 / 찌개용 등
        "detail":
            detail,

        # 다진 / 불린 / 마른 등
        "preparation":
            preparation,

        "group":
            group,

        # ingredient / seasoning
        "item_type":
            item_type,

        # 원본 양
        "amount_text":
            primary["amount_text"],

        # 범위
        "amount_min":
            primary["amount_min"],

        "amount_max":
            primary["amount_max"],

        "unit_raw":
            primary["unit_raw"],

        # T → 큰술 등
        "unit_normalized":
            primary["unit_normalized"],

        "qualitative_amount":
            qualitative_amount,

        # 미역 20g 1줌처럼 수량이 두 개 있는 경우
        "secondary_amounts":
            secondary_amounts,

        # "또는" 포함 여부
        "has_alternative":
            "또는" in name_text
            or "혹은" in name_text,

        # 반드시 보관
        "raw":
            raw,
    }


# ============================================================
# 6. 조리도구 추출
# ============================================================

def extract_tools_from_steps(
    steps
):

    found = {}


    for step in steps or []:

        if isinstance(
            step,
            dict
        ):

            text = (
                step.get("text")
                or ""
            )

            order = step.get(
                "order"
            )

        else:

            text = str(step)

            order = None


        for pattern, normalized_name in TOOL_PATTERNS:

            if re.search(
                pattern,
                text
            ):

                if normalized_name not in found:

                    found[
                        normalized_name
                    ] = {

                        "name":
                            normalized_name,

                        "source":
                            "steps_explicit",

                        "evidence_orders":
                            []
                    }


                if (
                    order is not None
                    and
                    order not in
                    found[
                        normalized_name
                    ]["evidence_orders"]
                ):

                    found[
                        normalized_name
                    ][
                        "evidence_orders"
                    ].append(
                        order
                    )


    return list(
        found.values()
    )


# ============================================================
# 7. 모음형 레시피 판별
# ============================================================

def is_collection_recipe(
    recipe
):

    title = (
        recipe.get("title")
        or ""
    )


    no_food_items = (

        not recipe.get(
            "ingredients"
        )

        and

        not recipe.get(
            "seasonings"
        )
    )


    collection_title = bool(
        re.search(
            r"\d+\s*가지|"
            r"모음집|"
            r"모음|"
            r"추천\s*메뉴|"
            r"메뉴\s*!?$",
            title
        )
    )


    return (
        no_food_items
        and
        collection_title
    )


# ============================================================
# 8. 인분 파싱
# ============================================================

def parse_range_number(
    text
):

    match = re.search(
        rf"({NUM})"
        rf"(?:\s*[~\-–]\s*({NUM}))?",
        text or ""
    )


    if not match:

        return (
            None,
            None
        )


    minimum = to_number(
        match.group(1)
    )


    if match.group(2):

        maximum = to_number(
            match.group(2)
        )

    else:

        maximum = minimum


    return (
        minimum,
        maximum
    )


def parse_servings(
    text
):

    raw = text or ""


    minimum, maximum = (
        parse_range_number(
            raw
        )
    )


    if minimum is None:

        return {

            "servings_raw":
                raw,

            "servings_min":
                None,

            "servings_max":
                None,
        }


    if "이상" in raw:

        maximum = None


    elif "이내" in raw:

        minimum = None


    return {

        "servings_raw":
            raw,

        "servings_min":
            minimum,

        "servings_max":
            maximum,
    }


# ============================================================
# 9. 조리시간 파싱
# ============================================================

def parse_cooking_time(
    text
):

    raw = text or ""


    match = re.search(
        rf"({NUM})"
        rf"(?:\s*[~\-–]\s*({NUM}))?"
        rf"\s*(시간|분)",
        raw
    )


    if not match:

        return {

            "cooking_time_raw":
                raw,

            "minutes_min":
                None,

            "minutes_max":
                None,
        }


    minimum = to_number(
        match.group(1)
    )


    if match.group(2):

        maximum = to_number(
            match.group(2)
        )

    else:

        maximum = minimum


    if match.group(3) == "시간":

        minimum *= 60

        maximum *= 60


    if "이상" in raw:

        maximum = None


    elif "이내" in raw:

        minimum = None


    return {

        "cooking_time_raw":
            raw,

        "minutes_min":
            minimum,

        "minutes_max":
            maximum,
    }


# ============================================================
# 10. Vector RAG용 원문 생성
# ============================================================

def build_document_text(
    recipe
):

    parts = []


    title = recipe.get(
        "title"
    )

    if title:

        parts.append(
            title
        )


    description = recipe.get(
        "description"
    )

    if description:

        parts.append(
            description
        )


    for step in (
        recipe.get("steps")
        or []
    ):

        if isinstance(
            step,
            dict
        ):

            text = step.get(
                "text"
            )

        else:

            text = str(step)


        if text:

            parts.append(
                text
            )


    return "\n".join(
        parts
    )


# ============================================================
# 11. 레시피 한 개 전체 전처리
# ============================================================

def preprocess_recipe(
    recipe
):

    ingredients_clean = []

    seasonings_clean = []


    # --------------------------------------------------------
    # 기존 ingredients + seasonings 모두 다시 검사
    # --------------------------------------------------------

    fields = [

        (
            "ingredient",
            "ingredients"
        ),

        (
            "seasoning",
            "seasonings"
        ),
    ]


    for original_type, field in fields:

        for item in (
            recipe.get(field)
            or []
        ):

            if isinstance(
                item,
                dict
            ):

                raw = (
                    item.get("raw")
                    or
                    " ".join(
                        str(x)
                        for x
                        in [
                            item.get("name"),
                            item.get("amount")
                        ]
                        if x
                    )
                )

                group = (
                    item.get("group")
                    or ""
                )

            else:

                raw = str(item)

                group = ""


            parsed = parse_food_item(
                raw,
                group,
                original_type
            )


            if not parsed[
                "name_normalized"
            ]:

                continue


            if (
                parsed["item_type"]
                == "seasoning"
            ):

                seasonings_clean.append(
                    parsed
                )

            else:

                ingredients_clean.append(
                    parsed
                )


    # --------------------------------------------------------
    # 정확히 같은 값 중복 제거
    # --------------------------------------------------------

    def dedupe(
        items
    ):

        seen = set()

        result = []


        for item in items:

            key = (

                item[
                    "name_normalized"
                ],

                item[
                    "amount_text"
                ],

                item[
                    "group"
                ],

                item[
                    "raw"
                ],
            )


            if key in seen:

                continue


            seen.add(
                key
            )

            result.append(
                item
            )


        return result


    ingredients_clean = dedupe(
        ingredients_clean
    )


    seasonings_clean = dedupe(
        seasonings_clean
    )


    # --------------------------------------------------------
    # 조리도구
    # --------------------------------------------------------

    tools_clean = (
        extract_tools_from_steps(
            recipe.get("steps")
            or []
        )
    )


    # --------------------------------------------------------
    # 모음형
    # --------------------------------------------------------

    is_collection = (
        is_collection_recipe(
            recipe
        )
    )


    # --------------------------------------------------------
    # 인분 / 시간
    # --------------------------------------------------------

    servings = parse_servings(
        recipe.get("servings")
    )


    cooking_time = (
        parse_cooking_time(
            recipe.get(
                "cooking_time"
            )
        )
    )


    # --------------------------------------------------------
    # 데이터 품질 경고
    # --------------------------------------------------------

    warnings = []


    if (
        not ingredients_clean
        and
        not seasonings_clean
    ):

        warnings.append(
            "no_food_items"
        )


    if not recipe.get(
        "steps"
    ):

        warnings.append(
            "no_steps"
        )


    if not recipe.get(
        "description"
    ):

        warnings.append(
            "no_description"
        )


    if not tools_clean:

        warnings.append(
            "no_explicit_tool"
        )


    if is_collection:

        warnings.append(
            "collection_candidate"
        )


    # --------------------------------------------------------
    # 기존 데이터 복사
    # --------------------------------------------------------

    result = dict(
        recipe
    )


    # --------------------------------------------------------
    # 통합 Recipe ID
    # --------------------------------------------------------

    result[
        "recipe_uid"
    ] = (

        f"{recipe.get('source', 'unknown')}_"
        f"{recipe.get('source_id', '')}"
    )


    result[
        "ingredients_clean"
    ] = ingredients_clean


    result[
        "seasonings_clean"
    ] = seasonings_clean


    result[
        "tools_clean"
    ] = tools_clean


    result[
        "is_collection"
    ] = is_collection


    # Neo4j 적재 후보
    result[
        "graph_eligible"
    ] = bool(

        recipe.get("title")

        and

        not is_collection

        and

        (
            ingredients_clean
            or
            seasonings_clean
        )
    )


    result[
        "preprocess_warnings"
    ] = warnings


    # Vector RAG용
    result[
        "document_text"
    ] = build_document_text(
        recipe
    )


    result.update(
        servings
    )


    result.update(
        cooking_time
    )


    return result

In [ ]:
from pathlib import Path

INPUT_DIR = Path(
    r"C:\Users\Playdata\Desktop\mle-01-p2-team2\홍기표\input"
)

OUTPUT_DIR = Path(
    r"C:\Users\Playdata\Desktop\mle-01-p2-team2\홍기표\output"
)

INPUT_FILE = INPUT_DIR / "recipes_10000.jsonl"

OUTPUT_FILE = OUTPUT_DIR / "recipes_10000_cleaned.jsonl"

NEO4J_FILE = OUTPUT_DIR / "recipes_10000_neo4j_ready.jsonl"

REPORT_FILE = OUTPUT_DIR / "preprocess_report.json"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

stats = Counter()

errors = []


with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as src, open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as out, open(
    NEO4J_FILE,
    "w",
    encoding="utf-8"
) as neo4j_out:


    for line_number, line in enumerate(
        src,
        start=1
    ):

        try:

            recipe = json.loads(
                line
            )


            cleaned = preprocess_recipe(
                recipe
            )


            # -----------------------------
            # 전체 전처리 결과
            # -----------------------------

            out.write(
                json.dumps(
                    cleaned,
                    ensure_ascii=False
                )
                + "\n"
            )


            stats[
                "total"
            ] += 1


            stats[
                "ingredients"
            ] += len(
                cleaned[
                    "ingredients_clean"
                ]
            )


            stats[
                "seasonings"
            ] += len(
                cleaned[
                    "seasonings_clean"
                ]
            )


            if cleaned[
                "tools_clean"
            ]:

                stats[
                    "recipes_with_tools"
                ] += 1


            if cleaned[
                "is_collection"
            ]:

                stats[
                    "collections"
                ] += 1


            # -----------------------------
            # Neo4j 적재 후보
            # -----------------------------

            if cleaned[
                "graph_eligible"
            ]:

                neo4j_out.write(
                    json.dumps(
                        cleaned,
                        ensure_ascii=False
                    )
                    + "\n"
                )

                stats[
                    "graph_eligible"
                ] += 1


            for warning in cleaned[
                "preprocess_warnings"
            ]:

                stats[
                    warning
                ] += 1


        except Exception as e:

            errors.append({

                "line":
                    line_number,

                "error":
                    str(e)
            })


            stats[
                "errors"
            ] += 1


report = {
    "stats":
        dict(stats),

    "errors":
        errors[:100]
}


with open(
    REPORT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report,
        f,
        ensure_ascii=False,
        indent=2
    )


print("전처리 완료")
print()

for key, value in stats.items():

    print(
        f"{key:25s}: {value:,}"
    )